In [1]:
import pandas as pd
import numpy as np
import joblib
from tensorflow import keras

model = keras.models.load_model("../models/project_risk_model.keras")
preprocessor = joblib.load("../models/preprocessor.pkl")

print("Model and preprocessor loaded successfully!")

Model and preprocessor loaded successfully!


In [3]:
# Load the original dataset
risk_df = pd.read_csv("../data/project_risk_raw_dataset.csv")

# Take one project as a sample
sample_project = risk_df.drop(
    columns=["Project_ID", "Risk_Level"]
).iloc[[0]].copy()

sample_project

,Project_Type,Team_Size,Project_Budget_USD,Estimated_Timeline_Months,Complexity_Score,Stakeholder_Count,Methodology_Used,Team_Experience_Level,Past_Similar_Projects,External_Dependencies_Count,...,Resource_Contention_Level,Industry_Volatility,Client_Experience_Level,Change_Control_Maturity,Risk_Management_Maturity,Team_Colocation,Documentation_Quality,Project_Start_Month,Current_Phase_Duration_Months,Seasonal_Risk_Factor
0,Construction,32,1526276.55,32,9.7,16,Waterfall,Senior,3,3,...,High,Extreme,First-time,Basic,Basic,Fully Colocated,Good,10,5,1.0


In [4]:
sample_processed = preprocessor.transform(sample_project)

print("Original features:", sample_project.shape)
print("Processed features:", sample_processed.shape)

Original features: (1, 49)
Processed features: (1, 120)


In [5]:
probabilities = model.predict(sample_processed)

class_names = ["Low", "Medium", "High", "Critical"]

predicted_index = np.argmax(probabilities[0])
predicted_class = class_names[predicted_index]
confidence = probabilities[0][predicted_index]

print("Predicted Risk:", predicted_class)
print(f"Confidence: {confidence:.2%}")

print("\nAll probabilities:")
for class_name, probability in zip(class_names, probabilities[0]):
    print(f"{class_name}: {probability:.2%}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 153ms/step
Predicted Risk: High
Confidence: 50.10%

All probabilities:
Low: 1.55%
Medium: 44.50%
High: 50.10%
Critical: 3.85%


In [6]:
actual_class = risk_df.iloc[0]["Risk_Level"]

print("Actual Risk:", actual_class)
print("Predicted Risk:", predicted_class)

if actual_class == predicted_class:
    print("✅ Correct prediction!")
else:
    print("❌ Incorrect prediction")

Actual Risk: High
Predicted Risk: High
✅ Correct prediction!


In [7]:
class_names = ["Low", "Medium", "High", "Critical"]

for i in range(5):
    sample_project = risk_df.drop(
        columns=["Project_ID", "Risk_Level"]
    ).iloc[[i]].copy()

    sample_processed = preprocessor.transform(sample_project)
    probabilities = model.predict(sample_processed, verbose=0)

    predicted_index = np.argmax(probabilities[0])
    predicted_class = class_names[predicted_index]
    actual_class = risk_df.iloc[i]["Risk_Level"]

    print(
        f"Project {i+1} | "
        f"Actual: {actual_class} | "
        f"Predicted: {predicted_class} | "
        f"Confidence: {probabilities[0][predicted_index]:.2%}"
    )

Project 1 | Actual: High | Predicted: High | Confidence: 50.10%
Project 2 | Actual: Low | Predicted: Low | Confidence: 98.57%
Project 3 | Actual: Medium | Predicted: Medium | Confidence: 67.52%
Project 4 | Actual: High | Predicted: High | Confidence: 80.83%
Project 5 | Actual: High | Predicted: High | Confidence: 89.69%


In [8]:
manual_project = risk_df.drop(
    columns=["Project_ID", "Risk_Level"]
).iloc[[0]].copy()

manual_project["Team_Size"] = 25
manual_project["Complexity_Score"] = 9
manual_project["Schedule_Pressure"] = 0.95
manual_project["Resource_Availability"] = 0.30
manual_project["Team_Turnover_Rate"] = 0.45

manual_processed = preprocessor.transform(manual_project)

probabilities = model.predict(manual_processed, verbose=0)

predicted_index = np.argmax(probabilities[0])

print("Predicted Risk:", class_names[predicted_index])

for name, prob in zip(class_names, probabilities[0]):
    print(f"{name}: {prob:.2%}")

Predicted Risk: Critical
Low: 0.00%
Medium: 0.05%
High: 11.34%
Critical: 88.61%


In [9]:
import tensorflow as tf

loaded_model = tf.saved_model.load(
    "../models/project_risk_savedmodel"
)

print("SavedModel loaded successfully!")
print(list(loaded_model.signatures.keys()))

SavedModel loaded successfully!
['serving_default']
